# 🌊 Python Graphs — BFS Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> BFS on a graph is like dropping a pebble in a pond.
> Ripples spread outward one ring at a time — every node at distance 1
> is visited before any node at distance 2.
> Use a queue and a visited set. Never re-visit a node.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [What Is Graph BFS? The Visual Model](#1) |
| 2 | [Graph Setup and Representations](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Number of Islands — LC 200](#5) |
| 6 | [Pattern 2: Rotting Oranges / Multi-Source BFS — LC 994](#6) |
| 7 | [Pattern 3: Word Ladder — LC 127](#7) |
| 8 | [Pattern 4: Walls and Gates — LC 286](#8) |
| 9 | [Pattern 5: Shortest Path in Binary Matrix — LC 1091](#9) |
| 10 | [The Graph BFS Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>

## 1. What Is Graph BFS? The Visual Model

```
GRAPH STRUCTURE

  Adjacency list:          Grid as graph:
  0 → [1, 2]               1 1 0       Cells are nodes.
  1 → [0, 3]               1 1 0       Edges connect 4-directional neighbors.
  2 → [0]                  0 0 1       '1' = land, '0' = water.
  3 → [1]

BFS RIPPLE — single source

  Source = node 0
  dist=0: {0}
  dist=1: {1, 2}  (neighbors of 0)
  dist=2: {3}     (neighbors of 1 not yet seen)

MULTI-SOURCE BFS

  Seed the queue with ALL sources at distance 0.
  Example: all rotten oranges start in the queue.
  They spread simultaneously — each step = 1 minute.

BFS ALWAYS FINDS SHORTEST PATH (unweighted graph).

KEY RULES:
  1. Use a queue (FIFO).
  2. Mark visited WHEN ENQUEUING, not when dequeuing.
     (Prevents adding the same node twice to the queue.)
  3. For grids: 4 directions = [(0,1),(0,-1),(1,0),(-1,0)]
```

<a id='2'></a>

## 2. Graph Setup and Representations

In [ ]:
from collections import deque, defaultdict
from typing import List, Dict

# ADJACENCY LIST — most common for BFS
edges = [(0,1),(0,2),(1,3),(2,3)]
graph = defaultdict(list)
for u, v in edges:
    graph[u].append(v)
    graph[v].append(u)       # undirected
print(f"adjacency list: {dict(graph)}")

# GRID AS GRAPH — 4 directions
DIRS = [(0,1),(0,-1),(1,0),(-1,0)]   # right, left, down, up

def valid(grid, r, c):
    return 0 <= r < len(grid) and 0 <= c < len(grid[0])

# BFS SKELETON
def bfs(graph, start):
    visited = {start}
    q = deque([start])
    dist = {start: 0}
    while q:
        node = q.popleft()
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)          # mark when enqueuing
                dist[neighbor] = dist[node] + 1
                q.append(neighbor)
    return dist

print(f"BFS distances from 0: {bfs(graph, 0)}")
print("Graph setup ready.")

<a id='3'></a>

## 3. The Core API — All Operations

```
OPERATION                              COMPLEXITY   WHAT IT DOES
──────────────────────────────────────────────────────────────────
BFS from single source                 O(V+E)       shortest path, visited set
Multi-source BFS (seed all starts)     O(V+E)       simultaneous spread
BFS on grid (4-dir or 8-dir)           O(m*n)       flood fill, shortest path
Count connected components             O(V+E)       outer loop + BFS each unvisited

THINGS YOU DO NOT DO:
❌  Mark visited when DEQUEUING — duplicates flood the queue
❌  Forget to check grid bounds before enqueuing
❌  Use BFS when the graph is weighted (use Dijkstra instead)
❌  Modify the grid while still needing original values (use visited set)
```

In [ ]:
# Demo: mark visited at ENQUEUE vs DEQUEUE — why it matters
from collections import deque

g = defaultdict(list)
for u, v in [(0,1),(0,2),(1,3),(2,3)]:
    g[u].append(v); g[v].append(u)

# CORRECT: mark at enqueue
def bfs_correct(start):
    visited = {start}
    q = deque([start])
    order = []
    while q:
        node = q.popleft()
        order.append(node)
        for nb in g[node]:
            if nb not in visited:
                visited.add(nb)      # mark HERE — prevents re-adding
                q.append(nb)
    return order

print(f"BFS order from 0: {bfs_correct(0)}")

# Demo: multi-source BFS
grid = [[2,1,1],[1,1,0],[0,1,1]]   # 2=rotten, 1=fresh
q2 = deque()
for r in range(len(grid)):
    for c in range(len(grid[0])):
        if grid[r][c] == 2:
            q2.append((r, c, 0))   # seed ALL rotten oranges at time 0
print(f"multi-source seeds: {list(q2)}")
print("Core API demo done.")

<a id='4'></a>

## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                    APPROACH
────────────────────────────────────────────────────────────────
Shortest path, unweighted graph          BFS from source
All nodes at same distance simultaneously Multi-source BFS
Count islands / connected components     BFS/DFS per unvisited cell
Word transformation min steps            BFS with string neighbors
Minimum hops / levels                    BFS, track level depth
Flood fill, spread from multiple starts  Multi-source BFS
Weighted shortest path                   Dijkstra (not plain BFS)
```

<a id='5'></a>

## 5. 🧩 Pattern 1: Number of Islands — LC 200

---

```
PROBLEM:
  Count connected groups of '1's in a 2D grid. Diagonal does NOT connect.

TRICK:
  Outer loop scans every cell. When '1' found, BFS/DFS to mark the whole
  island as visited, increment counter.

SLOW MOTION TRACE on:
  1 1 0
  1 1 0
  0 0 1

  r=0,c=0: '1' → BFS → visits (0,0),(0,1),(1,0),(1,1) → island 1
  r=0,c=1: already visited
  r=0,c=2: '0' skip
  r=2,c=2: '1' → BFS → visits (2,2) → island 2
  answer = 2

KEY INSIGHT:
  Mark visited by changing '1' to '0' (in-place) or use a visited set.
  Each BFS from an unvisited '1' = one complete island.

TIME:  O(m*n) — every cell visited at most twice
SPACE: O(m*n) — queue at most holds all cells
```

In [ ]:
def num_islands(grid: List[List[str]]) -> int:
    """
    LC 200 — Number of Islands
    Approach: BFS from each unvisited '1', mark visited in-place.
    Time:  O(m*n) — each cell processed at most twice
    Space: O(m*n) — queue in worst case (all land)
    """
    if not grid:
        return 0
    m, n = len(grid), len(grid[0])
    count = 0

    for r in range(m):
        for c in range(n):
            if grid[r][c] == '1':              # unvisited land
                count += 1
                q = deque([(r, c)])
                grid[r][c] = '0'               # mark visited at enqueue
                while q:
                    row, col = q.popleft()
                    for dr, dc in DIRS:
                        nr, nc = row+dr, col+dc
                        if valid(grid, nr, nc) and grid[nr][nc] == '1':
                            grid[nr][nc] = '0' # mark at enqueue
                            q.append((nr, nc))
    return count

# Slow motion on [[1,1,0],[1,1,0],[0,0,1]]:
# BFS from (0,0): sinks (0,0),(0,1),(1,0),(1,1) → count=1
# BFS from (2,2): sinks (2,2) → count=2

import copy
def test_harness(fn):
    tests = [
        ([['1','1','0'],['1','1','0'],['0','0','1']], 2),
        ([['1','1','1','1','0'],['1','1','0','1','0'],
          ['1','1','0','0','0'],['0','0','0','0','0']], 1),
        ([['1','0','0'],['0','0','0'],['0','0','1']], 2),
        ([['0']], 0),
        ([['1']], 1),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(copy.deepcopy(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(num_islands)
print("num_islands defined.")

<a id='6'></a>

## 6. 🧩 Pattern 2: Rotting Oranges / Multi-Source BFS — LC 994

---

```
PROBLEM:
  Grid has 0=empty, 1=fresh, 2=rotten. Every minute, rotten oranges
  rot adjacent fresh ones. Return minutes until all fresh are rotten, or -1.

TRICK:
  Multi-source BFS: seed queue with ALL rotten oranges at time=0.
  BFS naturally spreads level by level — each level = 1 minute.
  After BFS, if any fresh remain → return -1.

SLOW MOTION TRACE on [[2,1,1],[1,1,0],[0,1,1]]:
  seeds: (0,0) time=0
  minute 1: spread to (0,1),(1,0) → rotten
  minute 2: spread to (0,2),(1,1) → rotten
  minute 3: spread to (2,1) → rotten
  minute 4: spread to (2,2) → rotten
  answer=4

KEY INSIGHT:
  All sources start simultaneously — multi-source BFS simulates this.
  Time = levels of BFS - 1 (or track max time stored with each cell).

TIME:  O(m*n) — every cell processed once
SPACE: O(m*n) — queue holds all rotten cells
```

In [ ]:
def oranges_rotting(grid: List[List[int]]) -> int:
    """
    LC 994 — Rotting Oranges
    Approach: multi-source BFS from all rotten cells simultaneously.
    Time:  O(m*n) — each cell processed once
    Space: O(m*n) — queue
    """
    m, n = len(grid), len(grid[0])
    q = deque()
    fresh = 0

    # seed all rotten oranges at time 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 2:
                q.append((r, c, 0))    # (row, col, time)
            elif grid[r][c] == 1:
                fresh += 1

    time = 0
    while q:
        r, c, t = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r+dr, c+dc
            if valid(grid, nr, nc) and grid[nr][nc] == 1:
                grid[nr][nc] = 2       # mark rotten at enqueue
                fresh -= 1
                time = t + 1
                q.append((nr, nc, t+1))

    return time if fresh == 0 else -1

# Slow motion on [[2,1,1],[1,1,0],[0,1,1]]:
# seed: (0,0,0); spread reaches (2,2) at t=4
# answer=4

def test_harness(fn):
    tests = [
        ([[2,1,1],[1,1,0],[0,1,1]], 4),
        ([[2,1,1],[0,1,1],[1,0,1]], -1),
        ([[0,2]],                   0),
        ([[1,2]],                   1),
        ([[2]],                     0),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(copy.deepcopy(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(oranges_rotting)
print("oranges_rotting defined.")

<a id='7'></a>

## 7. 🧩 Pattern 3: Word Ladder — LC 127

---

```
PROBLEM:
  Transform beginWord to endWord changing one letter at a time.
  Each intermediate word must be in wordList. Return min transformations.

TRICK:
  BFS where each node is a word. Neighbors = all words differing by 1 char.
  Generate neighbors by trying all 26 substitutions at each position.
  Convert wordList to a set for O(1) lookup.

SLOW MOTION TRACE on begin='hit' end='cog' list=['hot','dot','dog','lot','log','cog']:
  level 1: hit
  level 2: hot (change h→h, i→o, t→t... hit→hot found in list)
  level 3: dot, lot
  level 4: dog, log
  level 5: cog → FOUND
  answer=5

KEY INSIGHT:
  Generating neighbors by character substitution avoids building
  the full graph upfront. 26 letters × word_length = bounded neighbor count.

TIME:  O(m² × 26) where m = word length, n = list size  ≈ O(m*n)
SPACE: O(n) — visited set + queue
```

In [ ]:
def ladder_length(beginWord: str, endWord: str, wordList: List[str]) -> int:
    """
    LC 127 — Word Ladder
    Approach: BFS, generate 1-char neighbors, O(1) lookup in word_set.
    Time:  O(m * n * 26) — m=word len, n=list size
    Space: O(n) — visited set
    """
    word_set = set(wordList)
    if endWord not in word_set:
        return 0

    q = deque([(beginWord, 1)])   # (word, steps)
    visited = {beginWord}

    while q:
        word, steps = q.popleft()
        for i in range(len(word)):
            for ch in 'abcdefghijklmnopqrstuvwxyz':
                neighbor = word[:i] + ch + word[i+1:]   # substitute one char
                if neighbor == endWord:
                    return steps + 1
                if neighbor in word_set and neighbor not in visited:
                    visited.add(neighbor)                # mark at enqueue
                    q.append((neighbor, steps + 1))

    return 0   # no transformation path found

# Slow motion on hit→cog:
# hit→hot(2)→dot,lot(3)→dog,log(4)→cog(5)

def test_harness(fn):
    tests = [
        ('hit','cog',['hot','dot','dog','lot','log','cog'], 5),
        ('hit','cog',['hot','dot','dog','lot','log'],       0),
        ('a',  'c',  ['a','b','c'],                         2),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(ladder_length)
print("ladder_length defined.")

<a id='8'></a>

## 8. 🧩 Pattern 4: Walls and Gates — LC 286

---

```
PROBLEM:
  Grid: -1=wall, 0=gate, INF=empty room.
  Fill each empty room with distance to nearest gate.

TRICK:
  Multi-source BFS: seed queue with ALL gates (value=0).
  BFS spreads outward, distance[neighbor] = distance[current] + 1.
  Only update rooms still at INF — they haven't been reached by a closer gate.

SLOW MOTION TRACE on:
  INF -1  0  INF
  INF INF INF  -1
  INF -1  INF INF
   -1 INF INF INF

  Seeds: (0,2) and... only one gate here.
  BFS spreads: (0,2)=0 → neighbors get 1 → their neighbors get 2 → ...
  Walls (-1) are never enqueued.

KEY INSIGHT:
  Multi-source from gates guarantees each room gets the MINIMUM distance.
  No need to BFS from each empty room separately — gates push outward.

TIME:  O(m*n) — each cell processed once
SPACE: O(m*n) — queue
```

In [ ]:
def walls_and_gates(rooms: List[List[int]]) -> None:
    """
    LC 286 — Walls and Gates
    Approach: multi-source BFS from all gates simultaneously.
    Modifies rooms in place.
    Time:  O(m*n) — each cell processed once
    Space: O(m*n) — queue
    """
    INF = 2147483647
    m, n = len(rooms), len(rooms[0])
    q = deque()

    # seed all gates
    for r in range(m):
        for c in range(n):
            if rooms[r][c] == 0:
                q.append((r, c))   # all gates start at distance 0

    while q:
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r+dr, c+dc
            if valid(rooms, nr, nc) and rooms[nr][nc] == INF:
                rooms[nr][nc] = rooms[r][c] + 1   # set distance
                q.append((nr, nc))                # mark by setting dist

# Slow motion: gate at (0,2)=0 → (0,1)=1, (0,3)=1, (1,2)=1 → ...

def test_harness(fn):
    INF = 2147483647
    tests = [
        ([[INF,-1,0,INF],[INF,INF,INF,-1],[INF,-1,INF,-1],[0,-1,INF,INF]],
         [[3,-1,0,1],[2,2,1,-1],[1,-1,2,-1],[0,-1,3,4]]),
        ([[0,-1],[INF,INF]],
         [[0,-1],[1,2]]),
        ([[INF]],
         [[INF]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        grid = copy.deepcopy(inputs[0])
        fn(grid)
        status = "PASSED" if grid == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={grid}")
        passed += (grid == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(walls_and_gates)
print("walls_and_gates defined.")

<a id='9'></a>

## 9. 🧩 Pattern 5: Shortest Path in Binary Matrix — LC 1091

---

```
PROBLEM:
  Find shortest clear path from top-left to bottom-right in binary matrix.
  '0' = clear, '1' = blocked. Move in 8 directions. Return path length or -1.

TRICK:
  BFS from (0,0), 8 directions. Track distance. First time you reach
  bottom-right = shortest path (BFS guarantee).

SLOW MOTION TRACE on [[0,1],[1,0]]:
  (0,0)=0 → neighbors: (0,1)=blocked, (1,0)=blocked, (1,1)=clear
  (0,0) → (1,1): distance=2
  answer=2

KEY INSIGHT:
  8-directional BFS (diagonal movement allowed) — extend DIRS to 8.
  Mark visited immediately when enqueuing to avoid re-processing.

TIME:  O(n²) for n×n grid
SPACE: O(n²) — queue
```

In [ ]:
DIRS8 = [(dr,dc) for dr in [-1,0,1] for dc in [-1,0,1] if (dr,dc) != (0,0)]

def shortest_path_binary_matrix(grid: List[List[int]]) -> int:
    """
    LC 1091 — Shortest Path in Binary Matrix
    Approach: BFS with 8-directional movement.
    Time:  O(n²) — each cell processed once
    Space: O(n²) — queue
    """
    n = len(grid)
    if grid[0][0] == 1 or grid[n-1][n-1] == 1:
        return -1                           # start or end is blocked
    if n == 1:
        return 1                            # single cell

    grid[0][0] = 1                          # mark start visited (value → distance)
    q = deque([(0, 0, 1)])                  # (row, col, distance)

    while q:
        r, c, dist = q.popleft()
        for dr, dc in DIRS8:
            nr, nc = r+dr, c+dc
            if 0 <= nr < n and 0 <= nc < n and grid[nr][nc] == 0:
                if nr == n-1 and nc == n-1:
                    return dist + 1         # found bottom-right
                grid[nr][nc] = 1           # mark visited
                q.append((nr, nc, dist+1))

    return -1

# Slow motion on [[0,1],[1,0]]:
# (0,0,1): only (1,1) reachable (diagonal) → return 1+1=2

def test_harness(fn):
    tests = [
        ([[0,1],[1,0]], 2),
        ([[0,0,0],[1,1,0],[1,1,0]], 4),
        ([[1,0,0],[1,1,0],[1,1,0]], -1),
        ([[0]], 1),
        ([[0,0],[0,0]], 2),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(copy.deepcopy(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(shortest_path_binary_matrix)
print("shortest_path_binary_matrix defined.")

<a id='10'></a>

## 10. The Graph BFS Decision Map

```
QUESTION TYPE                        PATTERN               LC
──────────────────────────────────────────────────────────────
Count connected components           BFS per unvisited     200
Simultaneous spread (all sources)    Multi-source BFS      994, 286
Word/state transformation min steps  BFS + neighbor gen    127
Min path in unweighted grid          BFS from source       1091
Level distances                      BFS, track depth      127, 102

SINGLE vs MULTI-SOURCE:
  One start point  → single-source BFS
  Multiple starts at t=0 → seed all in queue, multi-source BFS
  Multi-source naturally gives minimum distance to nearest source.
```

<a id='11'></a>

## 11. Interview Cheat Sheet

**1. When to use BFS:**

| Signal | Pattern |
|--------|--------|
| Shortest path, unweighted | Single-source BFS |
| All sources start together | Multi-source BFS |
| Min steps / hops | BFS, return depth |
| Count connected groups | BFS outer loop |

**2. Core templates — memorize these:**

```python
# TEMPLATE 1: SINGLE SOURCE BFS
visited = {start}
q = deque([(start, 0)])
while q:
    node, dist = q.popleft()
    for nb in graph[node]:
        if nb not in visited:
            visited.add(nb)       # mark at ENQUEUE
            q.append((nb, dist+1))

# TEMPLATE 2: MULTI-SOURCE BFS (grid)
q = deque()
for r in range(m):
    for c in range(n):
        if grid[r][c] == source_val:
            q.append((r,c))
            grid[r][c] = visited_val   # mark at enqueue
while q:
    r, c = q.popleft()
    for dr,dc in [(0,1),(0,-1),(1,0),(-1,0)]:
        nr,nc = r+dr, c+dc
        if in_bounds and condition:
            grid[nr][nc] = grid[r][c]+1
            q.append((nr,nc))
```

**3. Gotchas:**

```
❌  Mark visited at DEQUEUE — node enters queue multiple times
❌  Forget bounds check before grid access
❌  Use BFS for weighted graphs — use Dijkstra
✅  Multi-source = seed all starts → BFS spreads simultaneously
✅  BFS distance = minimum hops in unweighted graph (guaranteed)
```

<a id='12'></a>

## 12. Summary Map

```
GRAPH BFS
│
├── Single-Source BFS
│     ├── Shortest path from one node    LC 127
│     └── Flood fill, island detection   LC 200
│
├── Multi-Source BFS
│     ├── Seed all sources at t=0
│     ├── Rotting oranges spread         LC 994
│     └── Walls and gates distances      LC 286
│
└── Grid BFS
      ├── 4-directional (standard)       LC 200, 994
      └── 8-directional (diagonal ok)    LC 1091

GOLDEN RULE:
  Mark visited when ENQUEUING, not dequeuing.
  BFS = ripple outward = minimum distance guaranteed.
```

---
*End of Graphs BFS Master Guide — Sean Edition*